# VAE Training

Train the same small VAE on the curated, random, or mixed dataset. Set
`DATASET_SOURCE`, then run the notebook top-to-bottom.


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import random
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "convoy_sim").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader

from convoy_sim.vae import AttackProfileVAE, load_vae_dataset, vae_loss

PROJECT_ROOT

## Configuration

In [ ]:
DATASET_SOURCE = "curated"  # curated, random, or mixed

DATASETS = {
    "curated": (
        "train_random_tactical_v4_45k.jsonl",
        "valid_random_tactical_v4_5k.jsonl",
        "test_random_tactical_v4_5k.jsonl",
    ),
    "random": (
        "train_random_profile_v1_45k.jsonl",
        "valid_random_profile_v1_5k.jsonl",
        "test_random_profile_v1_5k.jsonl",
    ),
    "mixed": (
        "train_mixed_curated70_random30_45k.jsonl",
        "valid_mixed_curated70_random30_5k.jsonl",
        "test_mixed_curated70_random30_5k.jsonl",
    ),
}

DATA_DIR = PROJECT_ROOT / "data" / "attack_profiles" / "synthetic"
train_name, valid_name, test_name = DATASETS[DATASET_SOURCE]
TRAIN_PATH = DATA_DIR / train_name
VALID_PATH = DATA_DIR / valid_name
TEST_PATH = DATA_DIR / test_name if test_name else None

EPOCHS = 60
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
BETA = 0.03
LATENT_DIM = 6
HIDDEN_DIM = 64
SEED = 1945

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

random.seed(SEED)
torch.manual_seed(SEED)
DEVICE

## Load Data

In [ ]:
for path in (TRAIN_PATH, VALID_PATH, TEST_PATH):
    if path is not None and not path.exists():
        raise FileNotFoundError(f"Missing dataset: {path}")

train_data, preprocessor = load_vae_dataset(TRAIN_PATH)
valid_data, _ = load_vae_dataset(VALID_PATH, preprocessor)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)

print(f"train={len(train_data):,}  valid={len(valid_data):,}  device={DEVICE}")

## Train

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    totals = {"loss": 0.0, "recon_loss": 0.0, "kl_loss": 0.0}

    for batch in loader:
        batch = batch.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        recon, mu, logvar = model(batch)
        loss, stats = vae_loss(recon, batch, mu, logvar, beta=BETA)
        if training:
            loss.backward()
            optimizer.step()
        for name in totals:
            totals[name] += stats[name] * len(batch)

    return {name: total / len(loader.dataset) for name, total in totals.items()}

In [ ]:
run_name = datetime.now().strftime(f"%Y%m%d_%H%M%S_{DATASET_SOURCE}")
RUN_DIR = PROJECT_ROOT / "results" / "notebook-results" / "vae_train" / run_name
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "model_best.pt"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

model = AttackProfileVAE(latent_dim=LATENT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
history = []
best_valid_loss = float("inf")
started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    train_stats = run_epoch(model, train_loader, optimizer)
    with torch.inference_mode():
        valid_stats = run_epoch(model, valid_loader)

    row = {"epoch": epoch}
    row.update({f"train_{name}": value for name, value in train_stats.items()})
    row.update({f"valid_{name}": value for name, value in valid_stats.items()})
    history.append(row)

    if row["valid_loss"] < best_valid_loss:
        best_valid_loss = row["valid_loss"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "preprocessor": preprocessor.to_dict(),
                "hyperparameters": {"latent_dim": LATENT_DIM, "hidden_dim": HIDDEN_DIM},
            },
            CHECKPOINT_PATH,
        )

    print(
        f"epoch {epoch:03d} | train={row['train_loss']:.4f} "
        f"| valid={row['valid_loss']:.4f}"
    )

## Held-Out Test And Saved Results

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])

test_stats = None
if TEST_PATH is not None:
    test_data, _ = load_vae_dataset(TEST_PATH, preprocessor)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)
    with torch.inference_mode():
        test_stats = run_epoch(model, test_loader)
    print(f"held-out test={test_stats['loss']:.4f}")

elapsed_seconds = time.perf_counter() - started
history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / "training_history.csv", index=False)

metrics = {
    "dataset": {
        "source": DATASET_SOURCE,
        "train_samples": len(train_data),
        "valid_samples": len(valid_data),
        "test_samples": len(test_data) if TEST_PATH is not None else 0,
    },
    "training": {
        "best_valid_loss": best_valid_loss,
        "test_loss": test_stats["loss"] if test_stats else None,
        "seconds": elapsed_seconds,
    },
}
manifest = {
    "workflow": "vae_train_notebook",
    "dataset_source": DATASET_SOURCE,
    "train_path": str(TRAIN_PATH.relative_to(PROJECT_ROOT)),
    "valid_path": str(VALID_PATH.relative_to(PROJECT_ROOT)),
    "test_path": str(TEST_PATH.relative_to(PROJECT_ROOT)) if TEST_PATH else None,
    "preprocessor": preprocessor.to_dict(),
    "hyperparameters": {
        "latent_dim": LATENT_DIM,
        "hidden_dim": HIDDEN_DIM,
        "beta": BETA,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    },
}

(RUN_DIR / "metrics_summary.json").write_text(json.dumps(metrics, indent=2) + "\n")
(RUN_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(f"saved to {RUN_DIR.relative_to(PROJECT_ROOT)} in {elapsed_seconds / 60:.1f} minutes")

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["valid_loss"], label="validation")
axes[0].set(title="Total loss", xlabel="epoch")
axes[0].legend(frameon=False)

axes[1].plot(history_df["epoch"], history_df["train_recon_loss"], label="reconstruction")
axes[1].plot(history_df["epoch"], history_df["train_kl_loss"], label="KL")
axes[1].set(title="Training loss components", xlabel="epoch")
axes[1].legend(frameon=False)
plt.tight_layout()

In [ ]:
model.eval()
with torch.inference_mode():
    samples = model.sample(10, device=torch.device(DEVICE)).cpu()

sample_profiles = [
    preprocessor.decode_profile_fields(
        sample,
        profile_id=f"VAE_{index:04d}",
        name=f"vae_sample_{index:04d}",
    )
    for index, sample in enumerate(samples, start=1)
]

pd.DataFrame(sample_profiles)[
    ["profile_id", "u_pos", "base_bearing_rad", "spread_rad", "launch_delay_s"]
]